# 11 — Fixed-origin model benchmark

This notebook replaces the earlier one-fold leaderboard. The old global-ML results used teacher-forced lag features inside the 28-day holdout and are **retired**. They must not be cited as deployment performance.

The primary benchmark now uses multiple rolling origins and requires every model to forecast the complete holdout horizon using only information available at the forecast origin. Global ML therefore uses recursive prediction. The same contract is used by conformal calibration in the batch pipeline.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from clinic_forecast.backtesting import make_recursive_global_ml_adapter
from clinic_forecast.benchmark import benchmark_leaderboard, benchmark_metric_table, run_benchmark
from clinic_forecast.data import generate_network_data
from clinic_forecast.models.baseline import moving_average_forecast, seasonal_naive_forecast
from clinic_forecast.models.sarimax import sarimax_panel_forecast
from clinic_forecast.validation import RollingOriginSplitter

data_path = PROJECT_ROOT / 'data' / 'processed' / 'clinic_daily_usage.csv'
usage = pd.read_csv(data_path, parse_dates=['date']) if data_path.exists() else generate_network_data().usage
usage['date'] = pd.to_datetime(usage['date'])
print(usage.shape)


## Forecast contract

Each callable receives `(train, test)`, but `test.visits` is used only for scoring. The global-ML adapters explicitly discard target and same-day outcome columns before recursively forecasting the full horizon. Statistical and baseline models already forecast from the origin.


In [ ]:
EXOG = ['marketing_spend', 'campaign_active', 'is_holiday']
forecasters = {
    'seasonal_naive': lambda tr, te: seasonal_naive_forecast(train=tr, future=te),
    'moving_average_28': lambda tr, te: moving_average_forecast(train=tr, future=te, window=28),
    'sarimax_exog': lambda tr, te: sarimax_panel_forecast(tr, te, exog_cols=EXOG),
    'global_ml_hgb': make_recursive_global_ml_adapter('hgb'),
}

try:
    import xgboost  # noqa: F401
    forecasters['global_ml_xgboost'] = make_recursive_global_ml_adapter('xgboost')
except ImportError:
    pass

try:
    import lightgbm  # noqa: F401
    forecasters['global_ml_lightgbm'] = make_recursive_global_ml_adapter('lightgbm')
except ImportError:
    pass

print('Primary models:', list(forecasters))


In [ ]:
splitter = RollingOriginSplitter(
    initial_train_days=365 * 2,
    horizon_days=28,
    step_days=28,
    max_folds=4,
)
scored = run_benchmark(usage, forecasters, splitter, on_error='raise')
leaderboard = benchmark_leaderboard(scored)
leaderboard


In [ ]:
benchmark_metric_table(scored)


In [ ]:
output_dir = PROJECT_ROOT / 'reports' / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
leaderboard.to_csv(output_dir / 'model_benchmark_recursive.csv', index=False)
scored.to_csv(output_dir / 'model_benchmark_recursive_scored.csv', index=False)


## Interpretation

The production estimator should be chosen only from this deployment-matched multi-fold evidence. Optional deep-learning, foundation and API models remain useful supplementary experiments, but they should be added to this same fixed-origin harness before being compared with the primary models.
